# Evaluation of Splines
The obects that the class ``PeriodicSpline1D`` operates upon are mathematical functions and consist in descriptions of mappings $f:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f(x).$ Oftentimes, the class allows one to obtain a new *mapping* from an existing one, for instance the gradient of a spline (not just a number but the whole function $\dot{f}$) out of its original form $f.$ This is all good, but one is sometimes also interested in the more mundane goal of experimenting with a fixed mapping, typically to ask to what number $f(x)\in{\mathbb{R}}$ is the argument $x\in{\mathbb{R}}$ mapped to by $f:{\mathbb{R}}\rightarrow{\mathbb{R}}.$ The ``splinekit`` library  offers two possibilities to address this goal.
*   ``PeriodicSpline1D.at`` returns the value $f(x)$ of the spline $f$ evaluated at $x.$
*   ``PeriodicSpline1D.get_samples`` returns an array of values at arguments separated by a uniform step.

The two possibilities rely on the recipe followed by all one-dimensional uniform polynomial splines of nonnegative integer degree $n\in{\mathbb{N}},$ according to which an argument $x$ is mapped to the value

$$f(x)=\sum_{k\in{\mathbb{Z}}}\,c[{k\bmod K}]\,\beta^{n}(x-\delta x -k),$$
where $K\in{\mathbb{N}}+1$ is a positive integer period, $c$ is an arbitrary array of $K$ spline coefficients, $\beta^{n}$ is a B-spline whose degree $n$ is typeset in superscript (not a power), and $\delta x\in{\mathbb{R}}$ is an arbitrary delay. Put simply, a spline is a weighted sum of shifted B-splines.

Since the support of a B-spline is finite, the sum is finite at any given argument $x.$ Moreover, since B-splines of positive degrees are themselves made of unit-length pieces of polynomials, one can deploy the formalism of linear-algebra to make the expression of the spline recipe take the form

$$\begin{eqnarray*}
\forall n\in{\mathbb{N}}+1,\forall x\in{\mathbb{R}}:f(x)&=&\left(\begin{array}{c}c[{\left(-m\right)\bmod K}]\\c[{\left(1-m\right)\bmod K}]\\c[{\left(2-m]\right)\bmod K}\\\vdots\\c[{\left(n-m\right)\bmod K}]\end{array}\right)^{{\mathsf{T}}}\,\left(\begin{array}{ccccc}w_{0,0}^{n}&w_{0,1}^{n}&w_{0,2}^{n}&\cdots&w_{0,n}^{n}\\w_{1,0}^{n}&w_{1,1}^{n}&w_{1,2}^{n}&\cdots&w_{1,n}^{n}\\w_{2,0}^{n}&w_{2,1}^{n}&w_{2,2}^{n}&\cdots&w_{2,n}^{n}\\\vdots&\vdots&\vdots&\ddots&\vdots\\w_{n,0}^{n}&w_{n,1}^{n}&w_{n,2}^{n}&\cdots&w_{n,n}^{n}\end{array}\right)\,\left(\begin{array}{c}1\\v\\v^{2}\\\vdots\\v^{n}\end{array}\right)\\
&=&\color{blue}{{\mathbf{c}}^{{\mathsf{T}}}\,{\mathbf{W}}^{n}\,{\mathbf{v}}^{n}},
\end{eqnarray*}$$
where ${\mathbf{c}}\in{\mathbb{R}}^{n+1}$ is a vector whose $n+1$ components are extracted from the data-dependent array $c$ at some integer $m\in{\mathbb{Z}}$ that depends in discrete fashion on $\left(x-\delta x\right),$ where ${\mathbf{W}}\in{\mathbb{R}}^{\left(n+1\right)\times\left(n+1\right)}$ is a matrix that depends on $n$ only, and where ${\mathbf{v}}^{n}$ is a Vandermonde vector whose every component belongs to the interval $[0,1]$ and is made to depend continuously on $\left(x-\delta x\right).$ More detailed explanations are provided in the documentation of the functions
*   ``splinekit.spline_utilities._w_frac``
*   ``splinekit.bsplines.b_spline``
*   ``splinekit.splinekit.PeriodicSpline1D.at``


## Single Argument
To evaluate a one-dimensional periodic spline $f$ of positive degree $n$ at some argument $x$, it is enough to compute ${\mathbf{c}}^{{\mathsf{T}}}\,{\mathbf{W}}^{n}\,{\mathbf{v}}^{n}.$ This is precisely what is done by the function call ``f.at(x)`` in the piece of code below, which otherwise generates and plots a random spline of some arbitrary degree and delay. The part displayed between the two red stems corresponds to one period of the spline. For $n=0,$ the linear-algebra formalism breaks down and the computations made in ``f.at(x)`` revert to the explicit form $\sum_{k\in{\mathbb{Z}}}\,c[{k\bmod K}]\,\beta^{0}(x-\delta x -k).$

In [ ]:
# Load the required libraries.
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
period = 8 # Support of the data samples
max_degree = 5 # Maximal spline degree
max_delay = 5.0 # Maximal absolute delay

# Persistent spline
f = sk.PeriodicSpline1D()

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Widgets
degree = widgets.IntSlider(value = 3, min = 0, max = max_degree)
delay = widgets.FloatSlider(
    value = 0.0,
    min = -max_delay,
    max = max_delay,
    step = 0.125,
    readout_format = ".3f"
)
x = widgets.FloatSlider(
    value = 0.5 * period,
    min = -1.0,
    max = period + 1.0,
    step = 0.125,
    readout_format = ".3f",
    layout = widgets.Layout(width = "640px")
)
fx = widgets.Label(value = "")

# Plot
def update_plot (
    degree,
    delay,
    x
):
    global f # This spline

    # Check whether new data are needed
    if degree != f.degree:
        c = rng.standard_normal(period) # Fresh data
        f = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
    f.delay = delay
    fx.value = "$f({0:.3f}) = {1:.5f}$".format(x, f.at(x))
    # Plot canvas
    (fig, ax) = plt.subplots()
    # Spline in plain style
    f.plot(
        (fig, ax),
        plotpoints = 301,
        curve_markerfmt = " ",
        knot_marker = " ",
        periodbound_markerfmt = " "
    )
    ax.plot([x], [f.at(x)], "o")
    plt.show()

display(
    widgets.VBox([
        widgets.HBox([
            widgets.Label(value = "Degree", layout = widgets.Layout(width = "36px")),
            degree
        ]),
        widgets.HBox([
            widgets.Label(value = "Delay", layout = widgets.Layout(width = "36px")),
            delay
        ]),
        widgets.HBox([
            widgets.Label(value = "x", layout = widgets.Layout(width = "36px")),
            x
        ]),
        fx
    ]),
    widgets.interactive_output(update_plot, {'degree': degree, 'delay': delay, 'x': x})
)


## Multiple Arguments
We want now to evaluate $f(x)$ at the $L\,M$ arguments $x\in\left\{x_{0}+q/M\right\}_{q=0}^{L\,M-1},$ where $M\in{\mathbb{N}}+1$ is some positive integer oversampling factor, $L\in{\mathbb{N}}+1$ is the length of the support over which we want to sample $f,$ and $x_{0}\in{\mathbb{R}}$ is the argument of the first sample. To do so, we take advantage of ``PeriodicSpline1D.get_samples``.

In the next piece of code, we plot a random periodic spline of tunable period, degree, and delay; the domain of the plotted curve covers one period $[0,K),$  while the support of the plot is $[-5,K+5].$ Then, we overlay as solid blue dots the location of samples returned by ``get_samples(x0, support_length = L, oversampling = M)``, where $L$ is the number of blocks of samples, $M$ is the number of samples in a block, and $x_{0}$ gives the argument of the first sample. The locations returned by ``get_samples(x0, support_length = L)`` are shown in orange; they are a subset of ``get_samples(x0, support_length = L, oversampling = M)``.


In [ ]:
# Load the required libraries.
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
period = 8 # Period of the spline
max_period = 10 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 5.0 # Maximal absolute delay
max_support_length = 15 # Maximal number of blocks of samples
max_oversampling = 5 # Maximal number of samples per block
max_x0 = 5.0 # Maximal absolute argument of the initial sample

# Persistent spline
f = sk.PeriodicSpline1D()

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    L = 6,
    M = 3,
    x0 = 0.0
):
    global f # This spline

    # Check whether new data are needed
    if period != f.period or degree != f.degree:
        c = rng.standard_normal(period) # Fresh samples
        f = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
    f.delay = delay
    # Plot canvas
    (fig, ax) = plt.subplots()
    plt.xlim(-5.025, period + 5.025)
    # Spline in plain style
    f.plot(
        (fig, ax),
        plotpoints = 301,
        plotdomain = sk.interval.ClosedOpen((0, period)),
        curve_markerfmt = " ",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )
    # Array of spline values
    fM = f.get_samples(x0, support_length = L, oversampling = M)
    xM = np.array([x0 + q / M for q in range(M * L)])
    ax.plot(xM, fM, "o")
    f1 = f.get_samples(x0, support_length = L)
    x1 = np.array([x0 + q for q in range(L)])
    ax.plot(x1, f1, "oC1")
    plt.show()

# widgets.interactive(
widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    L = (1, max_support_length),
    M = (1, max_oversampling),
    x0 = (-max_x0, max_x0)
)


### Computational Efficiency
When we evaluate a spline jointly at multiple arguments spaced regularly as above, it is judicious to pay attention on how the terms of ${\mathbf{c}}^{{\mathsf{T}}}\,{\mathbf{W}}^{n}\,{\mathbf{v}}^{n}$ depend on the variables of interest. In particular, when we compare the computation of $f(x_{0}+q/M)$ to that of $f(x_{0}+\left(q+M\right)/M),$ a detailed analysis reveals that the vector ${\mathbf{W}}^{n}\,{\mathbf{v}}^{n}$ is identical in the two cases, which leads to computational savings. To ascertain their benefit, we propose to time the determination of regularly spaced spline samples through either repeated calls to ``PeriodicSpline1D.at`` or through a combined call to ``PeriodicSpline1D.get_samples``. We report in a table by how many times the combined approach is faster than the repeated independent calls. For simplicity, we sample a whole period and set $L=K.$
### Conclusion
The gain in computational efficiency increases with the oversampling factor and with the period of the spline. The degree of the spline does not play much of a role.

In [ ]:
# Load the required libraries
import math
import numpy as np
import time

import splinekit as sk # This library

# Experimental conditions
highest_degree = 9 # Number or tables
periods = [2, 5, 10, 20, 50, 100, 200, 500, 1000] # Period of the spline
highest_oversampling = 6 # Number of samples per unit length
repeats = 10 # Number of times one experiment is performed

# Initialize the generator of random numbers
rng = np.random.default_rng()

for degree in range(1, highest_degree + 1):
    # Table of results
    print()
    print("Acceleration for Spline Degree n =", degree)
    print("====================================================================")
    print("      Period |     2     5    10    20    50   100   200   500  1000")
    print("Oversampling |")
    print("-------------+------------------------------------------------------")
    for oversampling in range(1, highest_oversampling + 1):
        performance = [0.0]
        for period in periods:
            # Create a random spline
            f = sk.PeriodicSpline1D.from_spline_coeff(
                rng.standard_normal(period), # Fresh data
                degree = degree,
                delay = np.random.standard_normal() # Random delay
            )
            # Random starting points
            starting_points = rng.standard_normal(repeats)

            # Duration of "get_samples"
            start = time.perf_counter()
            for x0 in starting_points:
                f.get_samples(x0, support_length = period, oversampling = oversampling)
            end = time.perf_counter()
            get_samples_duration = end - start

            # Duration of repeated calls to "at"
            start = time.perf_counter()
            for x0 in starting_points:
                for q in range(period * oversampling):
                    f.at(x0 + q / oversampling)
            end = time.perf_counter()
            at_duration = end - start
            
            # Relative speed
            performance += [at_duration / get_samples_duration]
        print("M = {0:1d}        | {1:5.2f} {2:5.2f} {3:5.2f} {4:5.2f} {5:5.2f} {6:5.2f} {7:5.2f} {8:5.2f} {9:5.2f}".format(
            oversampling,
            performance[1],
            performance[2],
            performance[3],
            performance[4],
            performance[5],
            performance[6],
            performance[7],
            performance[8],
            performance[9]
        ))
    print("====================================================================")
